# Data Cleaning

This notebook focuses on preparing the dataset for exploratory data analysis by handling data quality issues identified during the data understanding stage.

## Objectives

The main objectives of this stage are:

- Remove unnecessary columns
- Handle placeholder values
- Clean company name and remove ratings from company name
- Clean and transform salary information
- Standardize job title information
- Save the cleaned dataset for further analysis

In [86]:
import pandas as pd
import numpy as np
import re

### Load Dataset

In [87]:
df = pd.read_csv('../data/raw/DataScientist.csv')

In [88]:
df.head()

,Unnamed: 0,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
0,0,0,Senior Data Scientist,$111K-$181K (Glassdoor est.),"ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...",3.5,Hopper\n3.5,"New York, NY","Montreal, Canada",501 to 1000 employees,2007,Company - Private,Travel Agencies,Travel & Tourism,Unknown / Non-Applicable,-1,-1
1,1,1,"Data Scientist, Product Analytics",$111K-$181K (Glassdoor est.),"At Noom, we use scientifically proven methods ...",4.5,Noom US\n4.5,"New York, NY","New York, NY",1001 to 5000 employees,2008,Company - Private,"Health, Beauty, & Fitness",Consumer Services,Unknown / Non-Applicable,-1,-1
2,2,2,Data Science Manager,$111K-$181K (Glassdoor est.),Decode_M\n\nhttps://www.decode-m.com/\n\nData ...,-1.0,Decode_M,"New York, NY","New York, NY",1 to 50 employees,-1,Unknown,-1,-1,Unknown / Non-Applicable,-1,True
3,3,3,Data Analyst,$111K-$181K (Glassdoor est.),Sapphire Digital seeks a dynamic and driven mi...,3.4,Sapphire Digital\n3.4,"Lyndhurst, NJ","Lyndhurst, NJ",201 to 500 employees,2019,Company - Private,Internet,Information Technology,Unknown / Non-Applicable,"Zocdoc, Healthgrades",-1
4,4,4,"Director, Data Science",$111K-$181K (Glassdoor est.),"Director, Data Science - (200537)\nDescription...",3.4,United Entertainment Group\n3.4,"New York, NY","New York, NY",51 to 200 employees,2007,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"BBDO, Grey Group, Droga5",-1


### Create a Working Copy

To preserve the original dataset, all preprocessing steps are applied to a copy of the data.

In [89]:
df_clean = df.copy()

### Remove Unnecessary Columns

The columns `Unnamed: 0` and `index` are row identifiers and do not contain meaningful information for the analysis.

In [90]:
df_clean = df_clean.drop(['Unnamed: 0', 'index'], axis=1)

In [91]:
df_clean.columns

Index(['Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors',
       'Easy Apply'],
      dtype='str')

### Replace placeholder values

In this section the columns containing placeholder value `-1` will be found and replaced by NaN value.

`-1` indicates unavialable information in this dataset and since it affects the statistical computations specially for numerical values it's better to be replaced by `Nan`.

In [92]:
(df_clean == -1).sum()

Job Title              0
Salary Estimate        0
Job Description        0
Rating               409
Company Name           0
Location               0
Headquarters           0
Size                   0
Founded              977
Type of ownership      0
Industry               0
Sector                 0
Revenue                0
Competitors            0
Easy Apply             0
dtype: int64

In [93]:
(df_clean == '-1').sum()

Job Title               0
Salary Estimate         0
Job Description         0
Rating                  0
Company Name            0
Location                0
Headquarters          240
Size                  229
Founded                 0
Type of ownership     229
Industry              546
Sector                546
Revenue               229
Competitors          2760
Easy Apply           3745
dtype: int64

Columns containing placeholder value `-1`:

- Numerical:
    - `Rating`
    - `Founded`

- String:
    - `Headquarters `
    - `Size`
    - `Type of ownership`
    - `Industry`
    - `Sector`
    - `Revenue`
    - `Competitors`
    - `Easy Apply`

In [94]:
df_clean = df_clean.replace(-1, np.nan)
df_clean = df_clean.replace('-1', np.nan)

In [95]:
(df_clean == -1).sum()

Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
Easy Apply           0
dtype: int64

In [96]:
(df_clean == '-1').sum()

Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
Easy Apply           0
dtype: int64

### Clean Company Name

The `Company Name` column contained company ratings appended to the end of many company names. Since the ratings are already available in the `Rating` column, this information is redundant.

To ensure consistent company names and avoid duplicate entries during analysis, the appended ratings will be removed, keeoing only the company name.

In [97]:
df_clean['Company Name'].sample(10)

172                        Mediaocean\n4.0
2638               Lucid Technologies\n3.9
3051                         Hiretual\n3.1
2702           Excite Health Partners\n3.8
1806                          TechUSA\n2.7
1348                 ON Semiconductor\n3.6
633                  Warner Media LLC\n3.7
2435                           Humana\n3.6
2963                      CyberCoders\n4.2
3380    University of Texas at Austin\n4.3
Name: Company Name, dtype: str

In [98]:
df_clean['Company Name'] = df_clean['Company Name'].str.split('\n').str[0].str.strip()

In [99]:
df_clean['Company Name'].value_counts().head(20)

Company Name
Apple                                60
IBM                                  57
Amazon                               47
Staffigo Technical Services, LLC     28
Facebook                             25
Diverse Lynx                         25
Solekai Systems Corp                 24
Southwest Research Institute         23
Varsity Tutors                       21
Humana                               20
CyberCoders                          18
GSK                                  18
J&J Family of Companies              17
Houston Methodist                    16
Pfizer                               16
Nationwide Children's Hospital       16
Kforce                               15
Spark Therapeutics                   15
Internal Revenue Service             14
The University of Texas at Austin    14
Name: count, dtype: int64

### Clean and Standardize Salary Estimate Column

The column `Salary Estimate` is stored as a string containing the range estimate of the salary.

In the part below the `Salary Estimate` column will be cleaned of any text content and two numbers of min-salary and max-salary will be extracted.

In [100]:
len(df_clean['Salary Estimate'].unique())

148

In [101]:
print(df_clean['Salary Estimate'].unique()[:100])
print(df_clean['Salary Estimate'].unique()[100:148])

<StringArray>
[    '$111K-$181K (Glassdoor est.)',     '$120K-$140K (Glassdoor est.)',
     '$119K-$147K (Glassdoor est.)',      '$74K-$124K (Glassdoor est.)',
     '$102K-$121K (Glassdoor est.)',     '$129K-$210K (Glassdoor est.)',
     '$156K-$254K (Glassdoor est.)',     '$116K-$152K (Glassdoor est.)',
     '$133K-$171K (Glassdoor est.)',     '$110K-$139K (Glassdoor est.)',
      '$82K-$141K (Glassdoor est.)',      '$96K-$119K (Glassdoor est.)',
     '$146K-$175K (Glassdoor est.)',     '$102K-$164K (Glassdoor est.)',
     '$113K-$180K (Glassdoor est.)',      '$76K-$122K (Glassdoor est.)',
     '$134K-$218K (Glassdoor est.)',      '$76K-$125K (Glassdoor est.)',
     '$132K-$208K (Glassdoor est.)',      '$83K-$105K (Glassdoor est.)',
      '$75K-$143K (Glassdoor est.)',       '$34K-$72K (Glassdoor est.)',
       '$37K-$75K (Glassdoor est.)',     '$113K-$184K (Glassdoor est.)',
     '$136K-$164K (Glassdoor est.)', '$34-$53 Per Hour(Glassdoor est.)',
       '$46K-$80K (Glassdoor est.)', 

In [102]:
df_clean['Salary Estimate'].str.contains('Per Hour', na=False).sum()

np.int64(21)

In [103]:
df_clean['Salary Estimate'].str.contains('K-', na=False).sum()

np.int64(3888)

In [104]:
df[df_clean['Salary Estimate'].str.contains('Per Hour', na=False)]['Salary Estimate']

685     $34-$53 Per Hour(Glassdoor est.)
686     $34-$53 Per Hour(Glassdoor est.)
687     $34-$53 Per Hour(Glassdoor est.)
688     $34-$53 Per Hour(Glassdoor est.)
689     $34-$53 Per Hour(Glassdoor est.)
690     $34-$53 Per Hour(Glassdoor est.)
691     $34-$53 Per Hour(Glassdoor est.)
1295    $10-$26 Per Hour(Glassdoor est.)
1296    $10-$26 Per Hour(Glassdoor est.)
1297    $10-$26 Per Hour(Glassdoor est.)
2124    $17-$27 Per Hour(Glassdoor est.)
2125    $17-$27 Per Hour(Glassdoor est.)
2126    $17-$27 Per Hour(Glassdoor est.)
2127    $17-$27 Per Hour(Glassdoor est.)
2128    $17-$27 Per Hour(Glassdoor est.)
2129    $17-$27 Per Hour(Glassdoor est.)
2130    $17-$27 Per Hour(Glassdoor est.)
2131    $17-$27 Per Hour(Glassdoor est.)
2132    $17-$27 Per Hour(Glassdoor est.)
2133    $17-$27 Per Hour(Glassdoor est.)
2134    $17-$27 Per Hour(Glassdoor est.)
Name: Salary Estimate, dtype: str

### Salary Format Handling

The `Salary Estimate` feature contains two different salary formats:

1. Annual salary estimates:
   - Example: `$93K-$151K (Glassdoor est.)`
   - Example: `$82K-$129K(Employer est.)`

2. Hourly salary estimates:
   - Example: `$34-$53 Per Hour(Glassdoor est.)`

Since these values use different units, hourly salaries are converted to estimated annual salaries using a standard assumption of 40 working hours per week and 52 weeks per year.

In [105]:
def extract_salary(salary):
    numbers = list(map(int, re.findall(r'\d+', salary)))
    min_salary, max_salary = tuple(numbers)

    if 'Per Hour' in salary:
        min_salary = round(min_salary * 40 * 52 / 1000)
        max_salary = round(max_salary * 40 * 52 / 1000)

    avg_salary = round((min_salary + max_salary) / 2)

    return pd.Series([min_salary, max_salary, avg_salary])

In [106]:
df_clean[['Min Salary (K)', 'Max Salary (K)', 'Average Salary (K)']] = (
    df_clean['Salary Estimate'].apply(extract_salary)
)

In [107]:
df_clean[['Min Salary (K)', 'Max Salary (K)', 'Average Salary (K)']].sample(20)

,Min Salary (K),Max Salary (K),Average Salary (K)
2907,111,175,143
3105,150,169,160
2736,45,78,62
1728,94,166,130
1490,47,73,60
2756,48,89,68
2389,75,149,112
2079,52,91,72
3821,52,94,73
1425,92,144,118


### Job Title Standardization

Job titles in the dataset are highly diverse and often describe the same role using different naming conventions.

To make role-based analysis easier while preserving the original information, two new features are created.

- **Job Category:** A standardized job family (e.g., Data Scientist, Data Engineer, Data Analyst).
- **Seniority:** A standardized seniority level (e.g., Junior, Senior, Lead, Principal, Mid).
- **Is Core Data Role:** Indicates whether the position belongs to a core data profession.

The original `Job Title` column is retained to preserve the raw information.

In [108]:
df_clean['Job Title'].unique()

<StringArray>
[                                              'Senior Data Scientist',
                                   'Data Scientist, Product Analytics',
                                                'Data Science Manager',
                                                        'Data Analyst',
                                              'Director, Data Science',
                                                      'Data Scientist',
                                             'Quantitative Researcher',
                                     'Quantitative Research Associate',
                                                        'AI Scientist',
                                     'Data Scientist/Machine Learning',
 ...
                                                     'Sr Data Modeler',
                    'Biotransformation Scientist and DMPK Design Lead',
 'JPSC-7975 - Data Analyst Lead- Columbus, OH (LOCAL CANDIDATES ONLY)',
               'Senior/Principal Scientist - 

In [109]:
df_clean['Job Title'].value_counts().head(60)

Job Title
Data Scientist                                                   274
Data Engineer                                                    260
Data Analyst                                                     246
Senior Data Scientist                                             91
Machine Learning Engineer                                         47
Senior Data Analyst                                               47
Big Data Engineer                                                 41
Business Intelligence Analyst                                     39
Senior Data Engineer                                              37
Lead Data Scientist                                               23
Sr. Data Scientist                                                23
Research Scientist                                                22
Data Modeler                                                      17
Principal Data Scientist                                          15
Junior Data Analyst     

In [110]:
df_clean['Job Title'] = df_clean['Job Title'].replace({'Data Sceintist': 'Data Scientist'})

In [111]:
df_clean["Job Title"] = df_clean["Job Title"].replace({'Sr. Big data Ebgineer': 'Sr. Big Data Engineer'})

In [112]:
## Assign a standardized job category based on keywords found in the original job title.
def get_job_category(title):
    title = title.lower()

    if 'data scientist' in title or 'data science' in title:
        return "Data Scientist"

    elif 'data engineer' in title:
        return 'Data Engineer'

    elif 'data analyst' in title:
        return 'Data Analyst'

    elif 'machine learning engineer' in title:
        return 'Machine Learning Engineer'

    elif 'business intelligence' in title:
        return 'Business Intelligence'

    elif 'research scientist' in title or 'ai scientist' in title:
        return 'Research Scientist'

    elif 'data modeler' in title:
        return 'Data Modeler'

    elif 'quantitative researcher' in title or 'quantitative research' in title:
        return 'Quantitative Research'

    elif 'analytics consultant' in title:
        return 'Analytics Consultant'

    elif 'quantitative analyst' in title or 'quantitative developer' in title:
        return 'Quantitative Analyst'

    elif 'computer scientist' in title:
        return 'Computer Scientist'

    elif 'bioinformatics scientist' in title:
        return 'Bioinformatics Scientist'

    elif 'machine learning scientist' in title:
        return 'Machine Learning Scientist'

    elif 'machine learning' and 'scientist' in title:
        return 'Machine Learning Scientist'

    elif 'machine learning' in title:
        return 'Machine Learning Engineer'

    elif 'physical scientist' in title:
        return 'Physical Scientist'

    elif 'data architect' in title or 'data analytics architect' in title:
        return 'Data Architect'

    elif 'molecular biology' in title:
        return 'Molecular Biology Scientist'

    elif 'statistician' in title:
        return 'Statistician'

    elif 'scientist' in title:
        return 'Scientist'

    elif "analytics consulting" in title or "analytic consultant" in title or "analytical consultant" in title or "data consultant" in title:
        return "Analytics Consultant"

    elif "big data consultant" in title:
        return "Data Engineer"

    elif "solution architect" in title and "data" in title:
        return "Data Architect"

    elif 'data visualization' in title:
        return 'Data Visualization'

    elif 'data infrastructure' in title or 'data platform' in title or 'data operations engineer' in title:
        return 'Data Engineer'

    elif 'analytics manager' in title or 'data analytics manager' in title:
        return 'Analytics Manager'

    elif 'data integration' in title or 'etl' in title or 'hadoop' in title:
        return 'Data Engineer'

    elif 'data software engineer' in title or 'software engineer' in title and 'data' in title:
        return 'Data Engineer'

    elif 'data visualization' in title:
        return 'Data Visualization'

    elif 'data reporting analyst' in title or 'reporting analyst' in title:
        return 'Data Analyst'

    elif 'data analytics insights analyst' in title or 'data production analyst' in title:
        return 'Data Analyst'

    elif 'data curator' in title or 'data custodian' in title:
        return 'Data Management'

    elif 'data manager' in title or 'head of data' in title:
        return 'Data Management'

    elif 'data analytics architect' in title or 'data intelligence architect' in title:
        return 'Data Architect'

    elif 'big data' in title or 'data crawler' in title or 'data & ml engineer' in title:
        return 'Data Engineer'

    elif 'master data management' in title or 'data management' in title:
        return 'Data Management'

    elif 'data intelligence architect' in title:
        return 'Data Architect'

    elif 'director of data' in title or 'vp' in title or 'data strategy' in title:
        return 'Data Leadership'

    elif 'data analytics manager' in title or 'data analytics and insights' in title:
        return 'Data Leadership'

    elif 'data intelligence' in title and 'architect' in title:
        return 'Data Architect'

    elif 'analytics consulting' in title:
        return 'Analytics Consultant'

    else:
        return 'Other'

In [113]:
## Extract the seniority level from the job title.
def get_seniority(title):
    title = title.lower()

    if 'director' in title:
        return 'Director'

    elif 'principal' in title:
        return 'Principal'

    elif 'lead' in title:
        return 'Lead'

    elif 'manager' in title:
        return 'Manager'

    elif 'senior' in title or re.search(r'\bsr\.?\b', title):
        return 'Senior'

    elif 'junior' in title or re.search(r'\bjr\.?\b', title):
        return 'Junior'
    else:
        return 'Unspecified'

In [114]:
## Return True if the job category belongs to a core data profession.
def is_core_data_role(category):
    core_roles = [
        'Data Scientist',
        'Data Engineer',
        'Data Analyst',
        'Machine Learning Engineer',
        'Machine Learning Scientist',
        'Business Intelligence',
        'Data Architect',
        'Data Modeler',
        'Statistician',
        'Quantitative Research',
        'Quantitative Analyst',
        'Data Visualization',
        'Research Scientist',
        'Analytics Consultant',
        'Analytics Manager',
        'Data Management'
    ]

    return category in core_roles

#### Apply get_job_category Function on Job Title

In [115]:
df_clean['Job Category'] = df_clean['Job Title'].apply(get_job_category)

In [116]:
df_clean['Job Category'].value_counts()

Job Category
Data Scientist                1155
Data Engineer                  762
Data Analyst                   749
Machine Learning Scientist     584
Research Scientist             133
Machine Learning Engineer      111
Business Intelligence           79
Analytics Consultant            61
Quantitative Research           52
Analytics Manager               49
Data Modeler                    39
Computer Scientist              34
Other                           31
Data Architect                  23
Quantitative Analyst            19
Bioinformatics Scientist        11
Data Management                 11
Data Leadership                  4
Data Visualization               2
Name: count, dtype: int64

In [117]:
d = df_clean[df_clean['Job Category'] == 'Other']
d['Job Title'].head(60)

215                     Intern- Martech & Data Consulting
260                         Director of Engineering, Data
481                  Exploratory Data Analysis Instructor
639     Senior Thermal Modeling Engineer(Relocation to...
1272    Statistics Consultants for Thesis, Math Tutors...
1497                     Senior Thermal Modeling Engineer
1545                   Principal Device Modeling Engineer
1555    Lead of Cybersecurity Data Protection & Analytics
1714                        Product Manager, Product Data
1832              Vertical Lift Data Exploitation Analyst
2134    Lead of Cybersecurity Data Protection & Analytics
2217                 System Performance Modeling Engineer
2276    Director, Data Partnering, Informatics and Pre...
2421         Sr. Computational Biologist, Immuno-Oncology
2432                 Clinical Research Data Specialist II
2433    Sr. Computational Biologist, Oncology Drug Dis...
2446      Computational Biologist/Bioinformatics Position
2540          

#### Apply get_seniority Function on Job Title

In [118]:
df_clean['Seniority'] = df_clean['Job Title'].apply(get_seniority)

In [119]:
df_clean['Seniority'].value_counts()

Seniority
Unspecified    2837
Senior          651
Principal       123
Lead            112
Manager         100
Junior           45
Director         41
Name: count, dtype: int64

In [120]:
d = df_clean[df_clean['Seniority'] == 'Mid']
d['Job Title'].head(60)

Series([], Name: Job Title, dtype: str)

#### Apply is_core_data_role Function on Job Category

In [121]:
df_clean['Is Core Data Role'] = df_clean['Job Category'].apply(is_core_data_role)

In [122]:
df_clean['Is Core Data Role'].value_counts()

Is Core Data Role
True     3829
False      80
Name: count, dtype: int64

In [123]:
df_clean[['Job Title', 'Job Category', 'Seniority', 'Is Core Data Role']].sample(10)

,Job Title,Job Category,Seniority,Is Core Data Role
860,Data Engineer,Data Engineer,Unspecified,True
3796,"Senior Scientist R&D - Nutrition -- Columbus, OH",Machine Learning Scientist,Senior,True
1042,Translational Bioinformatics Scientist,Bioinformatics Scientist,Unspecified,False
3873,"JPSC-6591 - ""Java/Hadoop Developer"" OR ""Big Da...",Data Engineer,Unspecified,True
2193,Data Analyst,Data Analyst,Unspecified,True
452,Data Scientist at IBM (Visa Sponsorship for Ca...,Data Scientist,Unspecified,True
2406,"Principal Scientist, Translational Pharmacology",Machine Learning Scientist,Principal,True
2188,Scientist,Machine Learning Scientist,Unspecified,True
2594,Data Analyst(Only W2),Data Analyst,Unspecified,True
1018,Senior Scientist-Impact Forecasting-Wild Fire ...,Machine Learning Scientist,Senior,True


### Feature Engineering

Two major feature engineering steps were performed:

1. **Salary Transformation**
    - Converted salary ranges into numerical values.
    - Standardized hourly salaries to annual salaries.
    - Created `Min Salary`, `Max Salary` and `Average Salary` for further analysis.
2. **Job Title Standardization**
    - Created `Job Category`
    - Created `Seniority`
    - Created `Is Core Data Role`

### Saving the Clean Dataset

The cleaned dataset will be saved and used for further analysis.

In [124]:
df_clean.to_csv('../data/processed/DataScientistClean.csv', index=False)

## Data Cleaning Summary

The dataset was cleaned and prepared for exploratory data analysis (EDA).

The main preprocessing steps included:

- Removed unnecessary columns.
- Replace placeholder values (`-1`) with missing values where appropriate.
- Standardized salary information and converted hourly salaries to annual salaries.
- Extracted numerical salary features (`Min Salary`, `Max Salary` and `Average Salary`).
- Standardized job titles into consistent categories.
- Extracted seniority levels from job titles.
- Identified core data-related roles.
- Saved the cleaned dataset for subsequent analusis.

The processed dataset is now ready for EDA.